# Task 1: Data Cleaning with Pandas

**What is data cleaning?** Real-world data is messy: some values are missing, some rows are repeated, and some numbers are stored as text. Before we can analyse anything, we fix these problems. That is data cleaning.

**What is pandas?** A Python library for working with tables of data. A table in pandas is called a **DataFrame** (we usually name it `df`).

**How to use this notebook:** Run each code cell with **Shift + Enter**, from top to bottom. Read the explanation above each cell first.

**Our dataset:** restaurant orders (bill amount, tip, day, etc.). Put `restaurant_tips_messy.csv` in the same folder as this notebook.

## Step 1: Import pandas and load the data
`import pandas as pd` loads the library (`pd` is a short nickname). `pd.read_csv()` reads a CSV file into a DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv("restaurant_tips_messy.csv")
df.head()   # shows the first 5 rows

## Step 2: Look at the data
Before cleaning, always look first. `shape` gives (rows, columns). `info()` shows each column's **data type** and how many values are not missing.

In [ ]:
print("Rows, Columns:", df.shape)
df.info()

**What to notice:** `total_bill` says `object` (text), but a bill amount should be a number. That is a **wrong data type** problem. Also, some columns have fewer than 251 non-null values, which means they have **missing values**.

## Step 3: Find the problems
`isnull().sum()` counts missing values in each column. `duplicated().sum()` counts rows that are exact copies of an earlier row.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## Step 4: Remove duplicates
Duplicate rows would count the same order twice and distort our results, so we drop them.

In [ ]:
df = df.drop_duplicates()
print("Rows after removing duplicates:", df.shape[0])

## Step 5: Fix the wrong data type
Some `total_bill` values look like `$16.99`. The `$` makes pandas treat the whole column as text. We:
1. remove the `$` with `.str.replace()`
2. convert to numbers with `pd.to_numeric()`

(`errors="coerce"` means: if something cannot be converted, make it missing instead of crashing.)

In [ ]:
df["total_bill"] = df["total_bill"].astype(str).str.replace("$", "", regex=False)
df["total_bill"] = pd.to_numeric(df["total_bill"], errors="coerce")
df["total_bill"].dtype   # should now say float64

## Step 6: Fix messy text
Some values look like `"  sun "` or `"male"`. Pandas sees `"Sun"` and `"  sun "` as different values. We fix this:
- `.str.strip()` removes extra spaces
- `.str.title()` makes capitalisation consistent (Sun, Male)

In [ ]:
for col in ["sex", "smoker", "day", "time"]:
    df[col] = df[col].str.strip().str.title()

df["day"].value_counts()   # now only 4 clean days

## Step 7: Handle missing values
We can either delete rows with missing values or **fill** them. With a small dataset, deleting loses too much, so we fill:
- **Numbers** are filled with the **median** (the middle value, which is not distorted by extreme values).
- **Categories** (text) are filled with the **mode** (the most common value).

In [ ]:
# Numbers -> median
df["total_bill"] = df["total_bill"].fillna(df["total_bill"].median())
df["tip"]        = df["tip"].fillna(df["tip"].median())
df["size"]       = df["size"].fillna(df["size"].median())

# Category -> most common value
df["sex"] = df["sex"].fillna(df["sex"].mode()[0])

# size is a count of people, so it should be a whole number
df["size"] = df["size"].astype(int)

print("Total missing values left:", df.isnull().sum().sum())

## Step 8: Check that everything is fixed

In [ ]:
print("Duplicates:", df.duplicated().sum())
print("Missing:", df.isnull().sum().sum())
df.info()

## Step 9: Basic statistics
- **mean** = the average
- **median** = the middle value
- **value_counts()** = how many times each category appears

In [ ]:
print("Mean tip:  ", round(df["tip"].mean(), 2))
print("Median tip:", df["tip"].median())
df[["total_bill", "tip", "size"]].describe()

In [ ]:
print(df["day"].value_counts())
print()
print(df["sex"].value_counts())

## Step 10: Save the clean data
We will use this clean file in Task 2 and Task 3.

In [ ]:
df.to_csv("restaurant_tips_clean.csv", index=False)
print("Saved!")

## Cleaning Report (your short summary for submission)

| Problem found | Fix applied | Why |
|---|---|---|
| 7 duplicate rows | `drop_duplicates()` | Avoid counting orders twice |
| `total_bill` stored as text (had `$`) | removed `$`, `pd.to_numeric()` | Numbers are needed for maths |
| Missing values in `total_bill`, `tip`, `sex`, `size` | median for numbers, mode for category | Keep all rows without distorting data |
| `size` was decimal | converted to `int` | People count is a whole number |
| Inconsistent text (`"  sun "`, `"male"`) | `strip()` + `title()` | Same category must look identical |

**Result:** 0 missing values, 0 duplicates, correct data types. Mean tip is about $2.99 and median tip is $2.83. Saturday is the busiest day.